# `dde` package demo

`dde` fits Deep Discrete Encoder (DDE) models: a hierarchy of $D$ binary latent layers
$A_D \to A_{D-1} \to \cdots \to A_1 \to Y$, where each layer is a sparse logistic
model of the layer above and the observed layer $Y$ follows a family-specific model.
A fitted model returns, for every layer, a coefficient matrix `B_list[d]` (column 0
is the intercept, columns 1.. are the loadings on the parent layer), the top-layer
Bernoulli probabilities `prop`, the latent states `A_list`, and, for the Normal
family, the residual variances `gamma`.

This notebook walks through the public interface on a simulated Normal DDE:

1. the kernel catalogue (what `fit()` can run);
2. simulating a small data set;
3. `fit()` with fixed latent widths and with CSP width selection;
4. `select_hyperparameters()` on an explicit grid and on the automatic grid;
5. how the same calls change for the other kernels.

Install the package in editable mode from the `dde/` directory (`pip install -e .`)
and run the notebook from there.

## 1. Implemented kernels

Every kernel is reached through the same call, `fit(Y, method=..., family=..., dimension_method=..., ...)`.
`method` picks the model class, `family` the observed-data distribution (DDE only), and
`dimension_method` whether the latent widths are supplied (`"fixed"`, pass `K_list`) or
selected by the cumulative shrinkage process (`"csp"`, omit `K_list`).

| method | family | widths | kernel | main hyperparameters |
|---|---|---|---|---|
| `DDE` | `Normal` | `fixed` | parametric SAEM with TLP penalty; closed-form observed layer for $D \ge 3$ | `K_list`, `pen_list`, `tau` |
| `DDE` | `Normal` | `csp` | copula-derived CSP kernel (`csp_backend="copula"`, the only Normal CSP kernel); runs on column-standardized $Y$ | `t_spike`, `temp`, `tau` |
| `DDE` | `Bernoulli` | `fixed` | parametric SAEM with TLP penalty; binary 0/1 data, no `gamma` | `K_list`, `pen_list`, `tau` |
| `DDE` | `Bernoulli` | `csp` | MCEM (default); `csp_backend="mcem_reinitialize"`, `"mcem_cap_only"`, or `"copula"` | `t_spike`, `temp`, `tau` |
| `DDE` | `Poisson` | `fixed` | parametric SAEM with TLP penalty; counts with a log link, no `gamma` | `K_list`, `pen_list`, `tau` |
| `DDE` | `Poisson` | `csp` | MCEM (default); `csp_backend="mcem_reinitialize"` or `"mcem_cap_only"` | `t_spike`, `temp`, `tau` |
| `SemiDDE` | – | `fixed` | semiparametric SAEM with an outer transformation (g) step; continuous data with unknown monotone margins | `K_list`, `pen_list`, `tau` |
| `SemiDDE` | – | `csp` | semiparametric SAEM with CSP shrinkage | `t_spike`, `temp`, `tau` |
| `CopDDE` | – | `fixed` | rank-likelihood SAEM (`get_SAEM_RL_D`); mixed-scale data | `K_list`, `pen_list`, `tau`, `temp` |
| `CopDDE` | – | `csp` | rank-likelihood CSP (`get_SAEM_RL_CSP_D`); the default CopDDE path | `t_spike`, `temp`, `tau` |

Hyperparameters shared by the kernels:

* `K_list`: latent widths from the shallowest layer to the deepest, e.g. `[6, 2]`. Fixed paths require it; CSP paths derive the maximal identifiable widths `floor(J/3), floor(floor(J/3)/3), ...` from the number of observed variables $J$ and shrink them.
* `pen_list`: one TLP penalty per layer (fixed paths). The package default is $N^{2/8}$ on every layer.
* `tau`: hard threshold applied to the loadings after each M-step; a loading below `tau` in absolute value is set to zero and defines the recovered graph. Default 0.3.
* `t_spike`: one spike scale per layer (CSP paths); smaller values shrink a layer more aggressively.
* `temp`: annealing temperature of the CSP latent updates (0.7 for $D \le 2$, 0.9 for deeper models by default; the discrete-family MCEM kernels usually need 0.9).
* `C`: Monte Carlo draws per SAEM iteration (1 in all studies), `tol`: convergence tolerance, `max_iter`: iteration cap, `random_state`: seed for the CSP paths.

In [1]:
import time
import numpy as np
import pandas as pd
from scipy.optimize import linear_sum_assignment

from dde import fit, select_hyperparameters
from dde.identifiability import build_true_params
from dde.simulation import generate_X_normal

np.set_printoptions(precision=3, suppress=True)
SEED = 2026

## 2. Simulated data

`build_true_params(K_list, J, max_val)` builds the sparse, identifiable loading
matrices used in the package's simulation studies (each latent variable has two
"anchor" observed variables plus a few cross loadings, intercepts $-2$ and $-4$,
top-layer probabilities 0.5). `generate_X_normal` then samples the latent layers
top-down and the Gaussian observations. The truth (`B_true`, `G_true`) is kept only
to score the fits below; with real data only `Y` exists.

The design has $N = 1500$ observations, $J = 18$ variables, $D = 2$ latent layers of
widths $(6, 2)$ and unit residual variances.

In [2]:
N, K_TRUE, D = 1500, [6, 2], 2
J = 3 * K_TRUE[0]
TAU = 0.3

prop_true, B_true, G_true, gamma_true = build_true_params(K_TRUE, J, 4.0)
prop_true = np.asarray(prop_true, float)
B_true = [np.asarray(B, float) for B in B_true]      # B_true[0]: (J, K1+1); B_true[1]: (K1, K2+1)
G_true = [np.asarray(G, bool) for G in G_true]       # true graphs (nonzero loadings)
gamma_true = np.asarray(gamma_true, float)

np.random.seed(SEED)
Y, A_true = generate_X_normal(N, prop_true, B_true, gamma_true)
print("Y shape:", Y.shape, " value range: [%.2f, %.2f]" % (Y.min(), Y.max()))
print("true widths:", K_TRUE, " true top-layer prop:", prop_true)
print("true observed-layer loadings B1 (first 6 rows):")
print(B_true[0][:6])

Y shape: (1500, 18)  value range: [-2.66, 3.16]
true widths: [6, 2]  true top-layer prop: [0.5 0.5]
true observed-layer loadings B1 (first 6 rows):
[[-0.866  1.732  0.     0.     0.     0.     0.   ]
 [-0.866  0.     1.732  0.     0.     0.     0.   ]
 [-0.512  0.     0.     1.977  0.     0.     0.   ]
 [-0.512  0.     0.     0.     1.977  0.     0.   ]
 [-0.743  0.     0.     0.     0.     1.753  0.   ]
 [-0.743  0.     0.     0.     0.     0.     1.753]]


The scoring helper below permutes the fitted latent columns to best match the truth
(latent variables are only identified up to column order), then reports the graph
accuracy (fraction of correctly recovered zero / nonzero loadings) and the coefficient
RMSE of every layer.

In [3]:
def align_to_truth(B_hat_list, B_true_list):
    B_hat = [np.asarray(B, float).copy() for B in B_hat_list]
    for layer, target in enumerate(B_true_list):
        if B_hat[layer].shape != target.shape:
            return None
        cost = np.sum((target[:, 1:].T[:, None, :] - B_hat[layer][:, 1:].T[None, :, :]) ** 2, axis=2)
        true_idx, fit_idx = linear_sum_assignment(cost)
        order = fit_idx[np.argsort(true_idx)]
        B_hat[layer][:, 1:] = B_hat[layer][:, 1:][:, order]
        if layer + 1 < len(B_hat):
            B_hat[layer + 1] = B_hat[layer + 1][order, :]
    return B_hat


def report(result, tau=TAU, seconds=None):
    print(f"fitted widths K_list = {list(result.K_list)}  (truth {K_TRUE})" + (f", {seconds:.1f}s" if seconds else ""))
    aligned = align_to_truth(result.B_list, B_true)
    if aligned is not None:
        acc = [float(np.mean((np.abs(B[:, 1:]) >= tau) == G)) for B, G in zip(aligned, G_true)]
        rmse = [float(np.sqrt(np.mean((B - T) ** 2))) for B, T in zip(aligned, B_true)]
        print(f"per-layer graph accuracy = {[round(a, 3) for a in acc]}, per-layer coefficient RMSE = {[round(r, 3) for r in rmse]}")
    print(f"top-layer prop = {np.round(result.prop, 3)}, mean residual variance gamma = {np.mean(result.gamma):.3f}")

## 3. `fit()` with fixed latent widths

With `dimension_method="fixed"` the widths are supplied through `K_list` and each layer
gets a TLP penalty from `pen_list`. The result is a `DDEResult` with fields
`K_list`, `prop`, `B_list`, `gamma`, `A_list`, `diagnostics` (iterations, convergence
metric, initialization, ...). `initialize="data"` (the default) uses the spectral
initializer; the kernel then runs penalized SAEM until the root-mean-square parameter
change drops below `tol` or `max_iter` is reached.

In [4]:
t0 = time.perf_counter()
fixed = fit(
    Y,
    method="DDE", family="Normal", D=D,
    dimension_method="fixed", K_list=K_TRUE,
    pen_list=[N ** 0.25] * D,   # package default penalty N^(2/8) on every layer
    tau=TAU, C=1, tol=0.005, max_iter=50,
    verbose=False,
)
report(fixed, seconds=time.perf_counter() - t0)
print("iterations:", fixed.diagnostics["iterations"])
print("B_list shapes:", [B.shape for B in fixed.B_list], " (column 0 = intercept)")
print("fitted observed-layer loadings B1 (first 6 rows, aligned to the truth):")
print(np.round(align_to_truth(fixed.B_list, B_true)[0][:6], 2))

fitted widths K_list = [6, 2]  (truth [6, 2]), 0.4s
per-layer graph accuracy = [1.0, 0.917], per-layer coefficient RMSE = [0.014, 0.368]
top-layer prop = [0.489 0.462], mean residual variance gamma = 0.250
iterations: 9
B_list shapes: [(18, 7), (6, 3)]  (column 0 = intercept)
fitted observed-layer loadings B1 (first 6 rows, aligned to the truth):
[[-0.85  1.71  0.    0.    0.    0.    0.  ]
 [-0.89  0.    1.78  0.    0.    0.    0.  ]
 [-0.52  0.    0.    2.    0.    0.    0.  ]
 [-0.51  0.    0.    0.    1.94  0.    0.  ]
 [-0.72  0.    0.    0.    0.    1.73  0.  ]
 [-0.73  0.    0.    0.    0.    0.    1.71]]


## 4. `fit()` with CSP width selection

With `dimension_method="csp"` the widths are not supplied. The kernel starts from the
maximal identifiable widths derived from $J$ and lets the cumulative shrinkage process
switch latent columns off; the active widths are returned in `result.K_list` and the
starting widths in `result.diagnostics["K_max_list"]`.

For the 18-variable design the maximal widths are already $(6, 2)$, so to show the
shrinkage we append 12 pure-noise variables: $J = 30$ gives starting widths $(10, 3)$,
and CSP has to recover $(6, 2)$ while the noise variables receive zero loadings.
`t_spike` holds one spike scale per layer and `temp` the annealing temperature.

In [5]:
rng = np.random.default_rng(1)
Y_csp = np.column_stack([Y, rng.normal(size=(N, 12))])   # 18 informative + 12 noise variables

t0 = time.perf_counter()
csp = fit(
    Y_csp,
    method="DDE", family="Normal", D=D,
    dimension_method="csp",          # no K_list
    t_spike=[0.03, 0.05], temp=0.7, tau=TAU,
    C=1, tol=0.005, max_iter=50, random_state=SEED,
    verbose=False,
)
print("starting widths K_max_list:", csp.diagnostics["K_max_list"], "->", "fitted widths:", list(csp.K_list))
B1_csp = np.asarray(csp.B_list[0])
print("loadings on the 12 noise variables all zero:", bool(np.all(B1_csp[18:, 1:] == 0)))
# score the informative rows against the truth
csp_informative = type(csp)(model=csp.model, K_list=csp.K_list, prop=csp.prop, gamma=csp.gamma[:18],
                            B_list=[B1_csp[:18]] + list(csp.B_list[1:]))
report(csp_informative, seconds=time.perf_counter() - t0)

starting widths K_max_list: (10, 3) -> fitted widths: [6, 2]
loadings on the 12 noise variables all zero: True
fitted widths K_list = [6, 2]  (truth [6, 2]), 4.2s
per-layer graph accuracy = [1.0, 0.917], per-layer coefficient RMSE = [0.015, 0.428]
top-layer prop = [0.518 0.485], mean residual variance gamma = 0.250


## 5. Hyperparameter selection

`select_hyperparameters` takes the same `method` / `family` / `D` / `dimension_method`
arguments as `fit`, fits every candidate, scores it, and (with `refit=True`, the
default) returns the model refitted with the best candidate. Scores:

* `criterion="ebic"` (recommended for DDE and SemiDDE): extended BIC computed on the full
  data, one fit per candidate;
* `"bic"` / `"loglik"`: K-fold hold-out scores (`n_splits` folds);
* `"pmse"` (CopDDE only): a cross-validated random-forest discrepancy between simulated
  and observed data.

Candidates can be given explicitly (rows, or axes crossed as a Cartesian grid; `per_layer`
crosses each layer's values independently). Fixed paths tune `pen_list` and `tau`; CSP
paths tune `t_spike` and `temp`. Everything else comes from `base_fit_kwargs` and the
package defaults. The scored table is in `result.diagnostics["cv_table"]`, and the
chosen row in `result.diagnostics["hyperparameter_selection"].best_candidate`.

In [6]:
t0 = time.perf_counter()
selected = select_hyperparameters(
    Y, method="DDE", family="Normal", D=D, dimension_method="fixed",
    criterion="ebic",
    fixed_hp={"K_list": K_TRUE},                                   # widths are not tuned here
    candidates={"pen_list": {"per_layer": [[N ** 0.25, N ** 0.375], [N ** 0.25]]},
                "tau": [0.2, 0.3, 0.4]},                            # 2 x 1 x 3 = 6 candidates
    base_fit_kwargs={"max_iter": 50, "C": 1, "verbose": False},
    random_state=SEED,
)
seconds = time.perf_counter() - t0
table = pd.DataFrame(selected.diagnostics["cv_table"])[["candidate_index", "pen_list", "tau", "mean_score", "fit_seconds"]]
table["pen_list"] = table["pen_list"].apply(lambda p: [round(x, 2) for x in p])
best = selected.diagnostics["hyperparameter_selection"].best_candidate
print(f"{len(table)} candidates scored in {seconds:.1f}s; selected:", {k: (round(v, 3) if isinstance(v, float) else [round(x, 2) for x in v]) for k, v in best.items()})
table

6 candidates scored in 1.8s; selected: {'pen_list': [15.53, 6.22], 'tau': 0.4}


,candidate_index,pen_list,tau,mean_score,fit_seconds
0,0,"[6.22, 6.22]",0.2,-16.632775,0.269003
1,1,"[6.22, 6.22]",0.3,-16.629152,0.231233
2,2,"[6.22, 6.22]",0.4,-16.624437,0.333996
3,3,"[15.53, 6.22]",0.2,-16.622479,0.358943
4,4,"[15.53, 6.22]",0.3,-16.621863,0.270305
5,5,"[15.53, 6.22]",0.4,-16.616222,0.289456


In [7]:
print("model refitted with the selected hyperparameters:")
report(selected, tau=float(best["tau"]))

model refitted with the selected hyperparameters:
fitted widths K_list = [6, 2]  (truth [6, 2])
per-layer graph accuracy = [1.0, 1.0], per-layer coefficient RMSE = [0.013, 0.421]
top-layer prop = [0.486 0.457], mean residual variance gamma = 0.250


When `candidates` is omitted the package builds its automatic grid: for fixed paths
`auto_grid` crosses layer-specific penalty exponents with threshold choices
($3 \cdot 2^D$ candidates); for DDE/SemiDDE CSP paths `auto_grid_copula_csp` crosses
three spike scales per layer with the temperatures $\{0.7, 0.9\}$ ($2 \cdot 3^D$
candidates). Each CSP row of `cv_table` also records the widths that candidate fitted
(`fitted_K`), which is useful for judging the selection.

In [8]:
t0 = time.perf_counter()
auto_fixed = select_hyperparameters(
    Y, method="DDE", family="Normal", D=D, dimension_method="fixed", criterion="ebic",
    fixed_hp={"K_list": K_TRUE}, base_fit_kwargs={"max_iter": 50, "C": 1, "verbose": False},
    random_state=SEED,
)
best = auto_fixed.diagnostics["hyperparameter_selection"].best_candidate
print(f"fixed path, automatic grid: {len(auto_fixed.diagnostics['cv_table'])} candidates in {time.perf_counter() - t0:.1f}s")
print("selected pen_list =", [round(p, 2) for p in best["pen_list"]], " tau =", round(best["tau"], 3))
report(auto_fixed, tau=float(best["tau"]))

fixed path, automatic grid: 12 candidates in 3.5s
selected pen_list = [15.53, 6.22]  tau = 0.321
fitted widths K_list = [6, 2]  (truth [6, 2])
per-layer graph accuracy = [1.0, 0.917], per-layer coefficient RMSE = [0.013, 0.34]
top-layer prop = [0.485 0.455], mean residual variance gamma = 0.250


In [9]:
t0 = time.perf_counter()
auto_csp = select_hyperparameters(
    Y_csp, method="DDE", family="Normal", D=D, dimension_method="csp", criterion="ebic",
    base_fit_kwargs={"max_iter": 50, "C": 1, "verbose": False},
    random_state=SEED,
)
best = auto_csp.diagnostics["hyperparameter_selection"].best_candidate
print(f"CSP path, automatic grid: {len(auto_csp.diagnostics['cv_table'])} candidates in {time.perf_counter() - t0:.1f}s")
print("selected t_spike =", best["t_spike"], " temp =", best["temp"], " -> fitted widths", list(auto_csp.K_list), "(truth", K_TRUE, ")")
csp_table = pd.DataFrame(auto_csp.diagnostics["cv_table"])[["candidate_index", "t_spike", "temp", "fitted_K", "mean_score"]]
csp_table.sort_values("mean_score", ascending=False).head(8)

CSP path, automatic grid: 18 candidates in 74.3s
selected t_spike = [0.01, 0.1]  temp = 0.9  -> fitted widths [6, 2] (truth [6, 2] )


,candidate_index,t_spike,temp,fitted_K,mean_score
11,11,"[0.01, 0.1]",0.9,"[6, 2]",-33.664122
5,5,"[0.03, 0.1]",0.7,"[6, 2]",-33.664269
15,15,"[0.05, 0.03]",0.9,"[6, 2]",-33.664304
4,4,"[0.03, 0.05]",0.7,"[6, 2]",-33.664660
17,17,"[0.05, 0.1]",0.9,"[6, 2]",-33.664735
8,8,"[0.05, 0.1]",0.7,"[6, 2]",-33.665215
0,0,"[0.01, 0.03]",0.7,"[6, 2]",-33.665757
9,9,"[0.01, 0.03]",0.9,"[6, 2]",-33.665762


## 6. Other kernels

The calls above change only in `method`, `family`, `dimension_method`, and the data
type. Typical settings used in the package's simulation studies:

```python
# Bernoulli DDE, binary 0/1 data; no gamma is returned
fit(Y_binary, method="DDE", family="Bernoulli", D=2, dimension_method="fixed",
    K_list=[6, 2], pen_list=[N ** 0.25] * 2, tau=0.3, C=1, tol=0.005, max_iter=50)
fit(Y_binary, method="DDE", family="Bernoulli", D=2, dimension_method="csp",
    t_spike=[0.03, 0.05], temp=0.9, tau=0.3, C=1, tol=0.005, max_iter=50, random_state=0)

# Poisson DDE, nonnegative integer counts
fit(Y_counts, method="DDE", family="Poisson", D=2, dimension_method="fixed",
    K_list=[6, 2], pen_list=[N ** 0.25] * 2, tau=0.3, C=1, tol=0.005, max_iter=50)
fit(Y_counts, method="DDE", family="Poisson", D=2, dimension_method="csp",
    t_spike=[0.03, 0.1], temp=0.9, tau=0.3, C=1, tol=0.005, max_iter=50, random_state=0)

# SemiDDE, continuous data with unknown monotone margins (outer g-step + inner SAEM)
fit(Y_continuous, method="SemiDDE", D=2, dimension_method="fixed", K_list=[6, 2],
    pen_list=[N ** 0.25] * 2, tau=0.3, C=1, tol=1.0, max_outer_iter=10, max_iter=50)
fit(Y_continuous, method="SemiDDE", D=2, dimension_method="csp",
    t_spike=[0.03, 0.05], temp=0.7, tau=0.3, C=1, tol=0.005, max_outer_iter=10, max_iter=50, random_state=0)

# CopDDE, mixed-scale data (rank likelihood); CSP is the default path
fit(Y_mixed, method="CopDDE", D=2, dimension_method="fixed", K_list=[6, 2],
    pen_list=[N ** 0.25] * 2, tau=0.3, temp=0.7, C=1, tol=0.005, max_iter=50, random_state=0)
fit(Y_mixed, method="CopDDE", D=2, dimension_method="csp",
    t_spike=[0.03, 0.05], temp=0.7, tau=0.3, C=1, max_iter=50, random_state=0)

# hyperparameter selection follows the same pattern, e.g. CopDDE with the pMSE score
select_hyperparameters(Y_mixed, method="CopDDE", D=2, dimension_method="csp", criterion="pmse",
                       n_splits=3, n_repeats=2, random_state=0)
```

Notes:

* The discrete-family MCEM CSP kernels (Bernoulli, Poisson) are sensitive to `temp`;
  0.9 keeps the top layer from collapsing on the simulation designs, whereas 0.7 can
  shrink it to zero.
* `tol` means different things per kernel: the postprocessed parameter-update RMSE for
  the parametric fixed paths, the Frobenius change scaled by $K_D/2$ for SemiDDE fixed,
  and a rolling standard deviation of the relative coefficient change for the CSP kernels.
* `method="auto"` (the default when `method` is omitted) picks SemiDDE for all-continuous
  data and CopDDE otherwise.
* `fit_dde_normal`, `fit_dde_bernoulli`, `fit_dde_poisson`, and the module-level kernels in
  `dde.algorithms` remain available for direct calls with the same keyword arguments.